## ASAP Cell Plots — Box Plots

Consolidated from three separate notebooks (Neurons, Microglia, Astrocytes). Set `cell_type` and `table_name` in the **Parameters** cell, then run all cells.

| `cell_type` | `table_name` | `ratio_regions_raw` |
|---|---|---|
| `'neurons'` | `'PCL_dens_percell_3D'` | `['cingulate', 'frontal', 'caudate']` |
| `'microglia'` | `'PCL_dens_percell_3D'` | `['cingulate', 'frontal', 'caudate']` |
| `'astrocytes'` | `'PCL_dens_percell_3D_pre052025'` | `['cingulate', 'frontal']` |

In [ ]:
# ============================================================
# PARAMETERS — edit these before running
# ============================================================
cell_type = "neurons"  # Options: 'neurons', 'microglia', 'astrocytes'

# Database table: use 'PCL_dens_percell_3D_pre052025' for astrocytes
table_name = "PCL_dens_percell_3D"

# Brain regions to include in the box plot (raw DB names, case-sensitive)
boxplot_regions_raw = ["cingulate", "frontal", "caudate"]

# Brain regions to include in the PD/Control ratio bar chart
# (omit 'caudate' for astrocytes: ['cingulate', 'frontal'])
ratio_regions_raw = ["cingulate", "frontal", "caudate"]

# Where to save output SVGs and statistics files
output_dir = r"/scratch/sycamore-asap/ASAP_Plots"

# Percentile filter (0 = include all data)
percentile = 0

In [ ]:
import duckdb
import seaborn as sns
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import sem
import numpy as np

sns.set_theme(style="ticks")
mm = 1 / 25.4  # millimetres in inches

import sys

sys.path.append("..")

from src import PlottingFunctions

plotter = PlottingFunctions.Plotter()

# Derived singular label used in axis/title text
_cell_label = {
    "neurons": "neuron",
    "microglia": "microglia",
    "astrocytes": "astrocyte",
}.get(cell_type, cell_type)

# Map raw DB region names to display names
_region_display = {
    "caudate": "Caudate",
    "cingulate": "Cingulate",
    "frontal": "Frontal",
}
boxplot_region_order = [
    _region_display[r] for r in boxplot_regions_raw if r in _region_display
]
ratio_region_order = [
    _region_display[r] for r in ratio_regions_raw if r in _region_display
]

# Connect to database
conn = duckdb.connect(r"/scratch/duckdb-database/main_survey_database.duckdb")

volume = (np.pi * (4 / 3)) * np.power(5, 3)

In [ ]:
# Function to create a boxplot with brain regions on x-axis
def create_region_boxplot(conn, y_metric="incelldens"):
    region_filter = ", ".join(f"'{r}'" for r in boxplot_regions_raw)
    query = f"""
    SELECT brain_region, {y_metric}, disease, donorid
    FROM {table_name}
    WHERE cell_type = '{cell_type}' AND percentile = {percentile}
          AND "area/um3" >= 100 AND "area/um3" <= 500
          AND brain_region IN ({region_filter})
    """
    subset = conn.execute(query).fetchdf()
    if subset.empty:
        print(f"No data for {cell_type} at {percentile}th percentile")
        return

    subset["brain_region"] = subset["brain_region"].map(_region_display)
    subset[y_metric] = subset[y_metric] * volume

    donor_medians = (
        subset.groupby(["brain_region", "disease", "donorid"])[y_metric]
        .median()
        .reset_index()
    )

    stats_output = [f"Statistical Testing ({y_metric})", "=" * 60]
    print(f"\nStatistical Testing ({y_metric}):")
    print("=" * 60)

    p_values = {}
    for region in boxplot_region_order:
        control_data = donor_medians[
            (donor_medians["brain_region"] == region)
            & (donor_medians["disease"] == "HC")
        ][y_metric].values
        pd_data = donor_medians[
            (donor_medians["brain_region"] == region)
            & (donor_medians["disease"] == "PD")
        ][y_metric].values
        if len(control_data) > 0 and len(pd_data) > 0:
            statistic, p_value = stats.mannwhitneyu(
                control_data, pd_data, alternative="two-sided"
            )
            p_values[region] = p_value
            sig = (
                "***"
                if p_value < 0.001
                else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns"
            )
            print(f"{region}: U={statistic:.2f}, p={p_value:.4f} {sig}")
            print(
                f"  Control: n={len(control_data)}, median={np.median(control_data):.3f}"
            )
            print(f"  PD: n={len(pd_data)}, median={np.median(pd_data):.3f}")
            stats_output += [
                f"\n{region}: Mann-Whitney U test (PD vs Control)",
                f"  U statistic = {statistic:.2f}",
                f"  p-value = {p_value:.4f} {sig}",
                f"  Control: n={len(control_data)}, median={np.median(control_data):.3f}, mean={np.mean(control_data):.3f}, SD={np.std(control_data, ddof=1):.3f}",
                f"  PD: n={len(pd_data)}, median={np.median(pd_data):.3f}, mean={np.mean(pd_data):.3f}, SD={np.std(pd_data, ddof=1):.3f}",
            ]
        else:
            p_values[region] = None
            print(f"{region}: Insufficient data")
            stats_output.append(f"\n{region}: Insufficient data")
    print("=" * 60)
    stats_output.append("\n" + "=" * 60)

    stats_filename = f"{output_dir}/{cell_type}_{percentile}percentile_{y_metric}_by_region_3D_boxplot_stats.txt"
    with open(stats_filename, "w") as f:
        f.write("\n".join(stats_output))
    print(f"Saved statistics to {stats_filename}")

    sns.set_theme(style="ticks")
    fig, ax = plotter.two_column_plot(height=(170 / 4) * mm, width=180 * mm, lw=0.75)

    palette = {"PD": "#E88247", "Control": "#C3B3A1"}
    hue_order = ["Control", "PD"]
    subset["disease"] = subset["disease"].replace("HC", "Control")

    ax = sns.boxplot(
        data=subset,
        x="brain_region",
        y=y_metric,
        hue="disease",
        hue_order=hue_order,
        palette=palette,
        showfliers=False,
        ax=ax,
        order=boxplot_region_order,
    )

    for i, region in enumerate(boxplot_region_order):
        region_data = subset[subset["brain_region"] == region]
        for j, disease in enumerate(hue_order):
            disease_data = region_data[region_data["disease"] == disease]
            donor_medians_plot = (
                disease_data.groupby("donorid")[y_metric].median().reset_index()
            )
            if donor_medians_plot.empty:
                continue
            num_hues = len(hue_order)
            offset = 0.4
            pos_offset = (j - (num_hues - 1) / 2) * (offset / num_hues)
            x_pos_base = i + pos_offset
            for median_val in donor_medians_plot[y_metric]:
                jitter = np.random.uniform(-0.05, 0.05)
                plt.scatter(
                    x_pos_base + jitter,
                    median_val,
                    color=palette[disease],
                    edgecolor="black",
                    s=30,
                    alpha=0.9,
                    zorder=10,
                )

    y_max = subset[y_metric].max()
    y_range = y_max - subset[y_metric].min()
    star_height = y_max + y_range * 0.05

    for i, region in enumerate(boxplot_region_order):
        if region in p_values and p_values[region] is not None:
            p_val = p_values[region]
            sig_text = (
                "***"
                if p_val < 0.001
                else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
            )
            text_fontsize = 8 if sig_text != "ns" else 6
            ax.plot([i - 0.2, i + 0.2], [star_height, star_height], "k-", lw=1)
            ax.text(
                i,
                star_height + y_range * 0.02,
                sig_text,
                ha="center",
                va="bottom",
                fontsize=text_fontsize,
                fontweight="bold",
            )

    ax.tick_params(labelsize=7)
    ax.get_legend().remove()

    legend_elements = []
    for region in boxplot_region_order:
        for disease in hue_order:
            d = subset[
                (subset["brain_region"] == region) & (subset["disease"] == disease)
            ]
            if not d.empty:
                unique_donors = d["donorid"].nunique()
                cell_count = len(d)
                short = {"Caudate": "CAUD", "Cingulate": "CIN", "Frontal": "FRO"}.get(
                    region, region[:4].upper()
                )
                label = f"{disease} {short} ($\\it{{N}}$ = {unique_donors}; $\\it{{n}}_{{cells}}$ = {cell_count:,})"
                legend_elements.append(
                    Patch(facecolor=palette[disease], edgecolor="black", label=label)
                )
    ax.legend(
        handles=legend_elements,
        loc="upper right",
        fontsize=6,
        ncols=1,
        bbox_to_anchor=(1.32, 1.05),
    )

    plt.grid(True, which="both", linestyle="--", linewidth=0.5, alpha=0.25)
    plt.xticks(rotation=0, ha="right")

    if y_metric == "puncta_cell_likelihood":
        plt.ylim(0, 10)

    if y_metric == "incelldens":
        plt.ylabel(f"\u03b1Syn oligomers per {_cell_label}", fontsize=8)
    elif y_metric == "puncta_cell_likelihood":
        plt.ylabel(f"Relative In-cell Density (spots/\u00b5m$^3$)", fontsize=8)
    else:
        plt.ylabel(y_metric, fontsize=8)
    ax.set(xlabel=None)

    filename = f"{output_dir}/{cell_type}_{percentile}percentile_{y_metric}_by_region_3D_boxplot.svg"
    plt.savefig(filename, dpi=1200, format="svg", bbox_inches="tight")
    plt.close()
    del subset
    print(f"Saved plot to {filename}")


create_region_boxplot(conn, "incelldens")

In [ ]:
# Function to create a PD/Control ratio bar chart with brain regions on x-axis
def calculate_ratio_plot(conn, y_metric="incelldens"):
    region_filter = ", ".join(f"'{r}'" for r in ratio_regions_raw)
    query = f"""
    SELECT brain_region, {y_metric}, disease, donorid
    FROM {table_name}
    WHERE cell_type = '{cell_type}' AND percentile = {percentile}
          AND "area/um3" >= 100 AND "area/um3" <= 500
          AND brain_region IN ({region_filter})
    """
    subset = conn.execute(query).fetchdf()
    if subset.empty:
        print(f"No data for {cell_type} at {percentile}th percentile")
        return

    subset["brain_region"] = subset["brain_region"].map(_region_display)
    subset["disease"] = subset["disease"].replace("HC", "Control")
    subset[y_metric] = subset[y_metric] * volume

    donor_medians = (
        subset.groupby(["brain_region", "disease", "donorid"])[y_metric]
        .median()
        .reset_index()
    )

    ratios, ratio_errors, n_donors, all_region_ratios = [], [], [], {}
    for region in ratio_region_order:
        control_vals = donor_medians[
            (donor_medians["brain_region"] == region)
            & (donor_medians["disease"] == "Control")
        ][y_metric].values
        pd_vals = donor_medians[
            (donor_medians["brain_region"] == region)
            & (donor_medians["disease"] == "PD")
        ][y_metric].values
        if len(control_vals) == 0 or len(pd_vals) == 0:
            ratios.append(np.nan)
            ratio_errors.append(np.nan)
            n_donors.append(0)
            all_region_ratios[region] = np.array([])
            continue
        control_median = np.median(control_vals)
        individual_ratios = pd_vals / control_median
        ratios.append(np.mean(individual_ratios))
        ratio_errors.append(np.std(individual_ratios, ddof=1))
        n_donors.append(len(pd_vals))
        all_region_ratios[region] = individual_ratios

    stats_output = [f"Statistical Testing for Ratios ({y_metric})", "=" * 60]
    print(f"\nStatistical Testing for Ratios ({y_metric}):")
    print("=" * 60)

    stats_output.append("\nSummary Statistics by Region:")
    print("\nSummary Statistics by Region:")
    for region in ratio_region_order:
        if len(all_region_ratios.get(region, [])) > 0:
            r = all_region_ratios[region]
            print(
                f"{region}: n={len(r)}, mean={np.mean(r):.3f}, median={np.median(r):.3f}, SD={np.std(r, ddof=1):.3f}, SEM={sem(r):.3f}"
            )
            stats_output += [
                f"\n{region}: n={len(r)}, mean={np.mean(r):.3f}, median={np.median(r):.3f}",
                f"  SD={np.std(r, ddof=1):.3f}, SEM={sem(r):.3f}, range=[{np.min(r):.3f}, {np.max(r):.3f}]",
            ]

    valid_ratios = [
        all_region_ratios[r]
        for r in ratio_region_order
        if len(all_region_ratios.get(r, [])) > 0
    ]
    valid_regions = [
        r for r in ratio_region_order if len(all_region_ratios.get(r, [])) > 0
    ]

    pairwise_results = []
    kw_p_value = None
    if len(valid_ratios) >= 2:
        h_stat, kw_p_value = stats.kruskal(*valid_ratios)
        sig_kw = "Significant" if kw_p_value < 0.05 else "No significant"
        print(
            f"\nKruskal-Wallis: H={h_stat:.3f}, p={kw_p_value:.4f} ({sig_kw} difference between regions)"
        )
        stats_output += [f"\nKruskal-Wallis H-test: H={h_stat:.3f}, p={kw_p_value:.4f}"]

        from itertools import combinations

        n_comparisons = len(list(combinations(valid_regions, 2)))
        bonferroni_alpha = 0.05 / n_comparisons
        print(f"Pairwise Mann-Whitney U (Bonferroni α={bonferroni_alpha:.4f}):")
        stats_output.append(
            f"Pairwise comparisons (Bonferroni-corrected α={bonferroni_alpha:.4f}):"
        )

        for r1, r2 in combinations(valid_regions, 2):
            stat, p_val = stats.mannwhitneyu(
                all_region_ratios[r1], all_region_ratios[r2], alternative="two-sided"
            )
            is_sig = p_val < bonferroni_alpha
            sig_text = (
                "***"
                if p_val < 0.001
                else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
            )
            pairwise_results.append((r1, r2, p_val, is_sig, sig_text))
            print(
                f"  {r1} vs {r2}: U={stat:.2f}, p={p_val:.4f} ({sig_text}), Bonferroni sig: {'Yes' if is_sig else 'No'}"
            )
            stats_output.append(
                f"  {r1} vs {r2}: U={stat:.2f}, p={p_val:.4f} ({sig_text}), sig: {'Yes' if is_sig else 'No'}"
            )
    else:
        print("Not enough regions with data for statistical comparison")

    print("=" * 60)
    stats_output.append("=" * 60)

    stats_filename = f"{output_dir}/{cell_type}_{percentile}percentile_{y_metric}_ratio_plot_stats.txt"
    with open(stats_filename, "w") as f:
        f.write("\n".join(stats_output))
    print(f"Saved statistics to {stats_filename}")

    sns.set_theme(style="ticks")
    fig, ax = plotter.two_column_plot(
        height=(170 / 4) * mm, width=(180 / 4) * mm, lw=0.75
    )

    x_pos = np.arange(len(ratio_region_order))
    ax.bar(
        x_pos,
        ratios,
        yerr=ratio_errors,
        color="#E88247",
        edgecolor="black",
        width=0.6,
        capsize=5,
        error_kw={"elinewidth": 1, "capthick": 1},
    )
    ax.axhline(y=1.0, color="#C3B3A1", linestyle="--", linewidth=1)

    if pairwise_results:
        y_max = max(r + e for r, e in zip(ratios, ratio_errors) if not np.isnan(r))
        region_to_x = {region: i for i, region in enumerate(ratio_region_order)}
        bracket_height_start = y_max * 1.15
        bracket_height_increment = y_max * 0.15
        sig_comparisons = [
            (r1, r2, sig) for r1, r2, p, is_sig, sig in pairwise_results if is_sig
        ]
        for idx, (r1, r2, sig_text) in enumerate(sig_comparisons):
            x1, x2 = region_to_x[r1], region_to_x[r2]
            bracket_y = bracket_height_start + idx * bracket_height_increment
            ax.plot(
                [x1, x1, x2, x2],
                [
                    bracket_y - 0.02 * y_max,
                    bracket_y,
                    bracket_y,
                    bracket_y - 0.02 * y_max,
                ],
                "k-",
                lw=1,
            )
            ax.text(
                (x1 + x2) / 2,
                bracket_y + 0.02 * y_max,
                sig_text,
                ha="center",
                va="bottom",
                fontsize=7,
                fontweight="bold",
            )
        if kw_p_value is not None and kw_p_value < 0.05:
            max_y = (
                bracket_height_start + len(sig_comparisons) * bracket_height_increment
            )
            ax.text(
                len(ratio_region_order) / 2 - 0.5,
                max_y + 0.1 * y_max,
                f"KW: p={kw_p_value:.3f}",
                ha="center",
                va="bottom",
                fontsize=6,
                style="italic",
            )

    ax.set_xticks(x_pos)
    ax.set_xticklabels(ratio_region_order)
    ax.tick_params(labelsize=7)
    for i, n in enumerate(n_donors):
        ax.text(x_pos[i], 0.1, f"n={n}", ha="center", va="bottom", fontsize=6)

    plt.grid(True, which="both", linestyle="--", linewidth=0.5, alpha=0.25)
    plt.ylabel("PD/Control ratio", fontsize=8)
    ax.set(xlabel=None)

    if y_metric == "incelldens":
        plt.title(f"Ratio of \u03b1Syn oligomers per {_cell_label}", fontsize=9)
    elif y_metric == "puncta_cell_likelihood":
        plt.title("Ratio of relative in-cell density", fontsize=9)

    filename = (
        f"{output_dir}/{cell_type}_{percentile}percentile_{y_metric}_ratio_plot.svg"
    )
    plt.savefig(filename, dpi=1200, format="svg", bbox_inches="tight")
    plt.close()
    print(f"Saved ratio plot to {filename}")


calculate_ratio_plot(conn, "incelldens")

# Close the database connection
conn.close()